# Explore ELIAS data

In [1]:
source("../RScripts/config.r")
source("../RScripts/plot_functions.r")
source("../RScripts/data_functions.r")
source("../RScripts/algo_functions.r")


options(repr.plot.width = 15, repr.plot.height = 10) # Adjust width & height


Linking to GEOS 3.11.1, GDAL 3.6.2, PROJ 9.1.1; sf_use_s2() is TRUE

Lade nötiges Paket: maps


Attache Paket: ‘rnaturalearthdata’


Das folgende Objekt ist maskiert ‘package:rnaturalearth’:

    countries110


Lade nötiges Paket: abind


Attache Paket: ‘dplyr’


Die folgenden Objekte sind maskiert von ‘package:stats’:

    filter, lag


Die folgenden Objekte sind maskiert von ‘package:base’:

    intersect, setdiff, setequal, union



Attache Paket: ‘purrr’


Das folgende Objekt ist maskiert ‘package:maps’:

    map



Attache Paket: ‘lubridate’


Das folgende Objekt ist maskiert ‘package:cowplot’:

    stamp


Die folgenden Objekte sind maskiert von ‘package:base’:

    date, intersect, setdiff, union



Attache Paket: ‘data.table’


Die folgenden Objekte sind maskiert von ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Das folgende Objekt ist maskiert ‘package:purrr’:

    transpose


Die folgenden Objekte sind maskiert von

In [10]:
path <- "/net/scratch/schoelleh96/WP2/WP2.2a/ELIAS_data"
ex_file <- "/2000/01/hit_20000101_12"

nc <- nc_open(paste0(path, ex_file))
elias <- ncvar_get(nc, "GT800p")
lon <- ncvar_get(nc, "lon")
lat <- ncvar_get(nc, "lat")
elias_t <- ncvar_get(nc, "GT800")


In [9]:
print(nc)


File /net/scratch/schoelleh96/WP2/WP2.2a/ELIAS_data/2000/01/hit_20000101_12 (NC_FORMAT_CLASSIC):

     6 variables (excluding dimension variables):
        float GT800p[lon,lat]   
        float MIDTROPp[lon,lat]   
        float LT400p[lon,lat]   
        float GT800[lon,lat]   
        float MIDTROP[lon,lat]   
        float LT400[lon,lat]   

     2 dimensions:
        lat  Size:96 
        lon  Size:360 


In [2]:
wcb_clim <- readRDS("../data/wr_wcb_clim.rds")


In [3]:
str(wcb_clim)


List of 13
 $ lon          : num [1:121(1d)] -80 -79 -78 -77 -76 -75 -74 -73 -72 -71 ...
 $ lat          : num [1:60(1d)] 30 31 32 33 34 35 36 37 38 39 ...
 $ lag_names    : chr [1:3] "lag0" "lag12" "lag24"
 $ lag_hours    : int [1:3] 0 12 24
 $ regimes      : chr [1:8] "AT" "ZO" "ScTr" "AR" ...
 $ variables    : chr [1:3] "inflow" "ascent" "outflow"
 $ nc_variables : chr [1:3] "GT800" "MIDTROP" "LT400"
 $ climatology  : num [1:121, 1:60, 1:3, 1:8, 1:3] 0.0734 0.0834 0.0935 0.103 0.103 ...
  ..- attr(*, "dimnames")=List of 5
  .. ..$ : NULL
  .. ..$ : NULL
  .. ..$ : chr [1:3] "lag0" "lag12" "lag24"
  .. ..$ : chr [1:8] "AT" "ZO" "ScTr" "AR" ...
  .. ..$ : chr [1:3] "inflow" "ascent" "outflow"
 $ hit_counts   : num [1:121, 1:60, 1:3, 1:8, 1:3] 109 124 139 153 153 157 145 149 144 140 ...
  ..- attr(*, "dimnames")=List of 5
  .. ..$ : NULL
  .. ..$ : NULL
  .. ..$ : chr [1:3] "lag0" "lag12" "lag24"
  .. ..$ : chr [1:8] "AT" "ZO" "ScTr" "AR" ...
  .. ..$ : chr [1:3] "inflow" "ascent" "out

In [8]:
nc <- nc_open("../data/geopotential.nc")

lon_z <- ncvar_get(nc, "longitude")
lat_z <- ncvar_get(nc, "latitude")
time_data <- ncvar_get(nc, "valid_time")

time_origin <- sub(
    "^seconds since ",
    "",
    ncatt_get(nc, "valid_time", "units")$value
)

z_time <- as.POSIXct(time_data, origin = time_origin, tz = "UTC")

z <- ncvar_get(nc, "z")
nc_close(nc)


In [9]:
nc <- nc_open("../data/geopotential00.nc")
time_data <- ncvar_get(nc, "valid_time")
z_time_00 <- as.POSIXct(time_data, origin = time_origin, tz = "UTC")
z_00 <- ncvar_get(nc, "z")
nc_close(nc)


In [12]:
z_all <- abind::abind(z, z_00, along = 3)
time_all <- c(z_time, z_time_00)


In [14]:
str(time_all)


 POSIXct[1:62094], format: "1940-01-01 12:00:00" "1940-01-02 12:00:00" "1940-01-03 12:00:00" ...


In [15]:
calculate_wr_lagged_z_composites <- function(
    z_data,
    z_times,
    lon,
    lat,
    wr_data,
    lag_hours,
    lag_names = paste0("lag", lag_hours)) {
    stopifnot(length(lag_hours) == length(lag_names))

    # Keep times as POSIXct
    z_times <- as.POSIXct(z_times, tz = "UTC")

    wr_dt <- data.table::as.data.table(wr_data)
    wr_dt[, time := as.POSIXct(time, tz = "UTC")]

    # sort lon/lat explicitly
    o_lon <- order(lon)
    o_lat <- order(lat)

    lon2 <- lon[o_lon]
    lat2 <- lat[o_lat]
    z2 <- z_data[o_lon, o_lat, , drop = FALSE]

    z_lookup <- data.table::data.table(
        time = z_times,
        time_idx = seq_along(z_times)
    )

    out <- list()
    k <- 1

    regimes <- unique(wr_dt[, .(wrindex, wrname)])

    for (j in seq_along(lag_hours)) {
        lag_h <- lag_hours[j]
        lag_nm <- lag_names[j]

        wr_shift <- data.table::copy(wr_dt)
        wr_shift[, target_time := time - lag_h * 3600]

        merged <- merge(
            wr_shift[, .(target_time, wrindex, wrname)],
            z_lookup,
            by.x = "target_time",
            by.y = "time"
        )

        for (i in seq_len(nrow(regimes))) {
            wr <- regimes$wrindex[i]
            wname <- regimes$wrname[i]

            idx <- merged[wrindex == wr, time_idx]
            if (!length(idx)) next

            mean_m <- apply(
                z2[, , idx, drop = FALSE],
                c(1, 2),
                mean,
                na.rm = TRUE
            )

            grid <- expand.grid(
                lon = lon2,
                lat = lat2
            )

            out[[k]] <- data.table::data.table(
                lon = grid$lon,
                lat = grid$lat,
                z = as.vector(mean_m),
                wrindex = wr,
                wrname = wname,
                lag_name = lag_nm,
                lag_hours = lag_h
            )

            k <- k + 1
        }
    }

    data.table::rbindlist(out, fill = TRUE)
}


In [16]:
result <- wrera(
    start = "19500111_00",
    end = "20250113_21",
    hours = c("12"),
    tformat = "string",
    setup = "z500anom_1979_2019_on_wrdef_10d_1.0_1979_2019",
    dataset = "era5",
    basepath = "../WR_read_example_package/wr_era5_update_1950_latwgt/"
)

wr_df <- result$data$LC
wr_df$date <- as.Date(wr_df$time)
str(wr_df)


Classes ‘data.table’ and 'data.frame':	27397 obs. of  5 variables:
 $ tsince : int  -253956 -253932 -253908 -253884 -253860 -253836 -253812 -253788 -253764 -253740 ...
 $ time   : POSIXct, format: "1950-01-11 12:00:00" "1950-01-12 12:00:00" ...
 $ wrindex: Factor w/ 8 levels "1","6","7","2",..: 8 8 8 8 8 8 5 5 5 5 ...
 $ wrname : Factor w/ 8 levels "AT","ZO","ScTr",..: 8 8 8 8 8 8 5 5 5 5 ...
 $ date   : Date, format: "1950-01-11" "1950-01-12" ...
 - attr(*, ".internal.selfref")=<externalptr> 


In [17]:
z_comp <- calculate_wr_lagged_z_composites(
    z_data = z_all,
    z_times = time_all,
    lon = lon_z,
    lat = lat_z,
    wr_data = wr_df,
    lag_hours = wcb_clim$lag_hours,
    lag_names = wcb_clim$lag_names
)


In [18]:
str(z_comp)
z_comp |>
    dplyr::count(lag_name)


Classes ‘data.table’ and 'data.frame':	699864 obs. of  7 variables:
 $ lon      : num  -80 -79.5 -79 -78.5 -78 -77.5 -77 -76.5 -76 -75.5 ...
 $ lat      : num  30 30 30 30 30 30 30 30 30 30 ...
 $ z        : num  57112 57115 57118 57122 57125 ...
 $ wrindex  : Factor w/ 8 levels "1","6","7","2",..: 8 8 8 8 8 8 8 8 8 8 ...
 $ wrname   : Factor w/ 8 levels "AT","ZO","ScTr",..: 8 8 8 8 8 8 8 8 8 8 ...
 $ lag_name : chr  "lag0" "lag0" "lag0" "lag0" ...
 $ lag_hours: int  0 0 0 0 0 0 0 0 0 0 ...
 - attr(*, ".internal.selfref")=<externalptr> 


lag_name,n
<chr>,<int>
lag0,233288
lag12,233288
lag24,233288


In [19]:
plot_composite_map_grid <- function(
    df,
    vars,
    row_labels,
    col_labels,
    row_order = NULL,
    col_order = NULL,
    contour_var = NULL,
    contour_binwidth = 10,
    colorbar_title = "Mean correlation $r$",
    value_name = "value",
    tile_width = NULL,
    tile_height = NULL) {
    make_lonlat_crop <- function(lon_bound, lat_bound, n = 1000) {
        lon_seq <- seq(lon_bound[1], lon_bound[2], length.out = n)
        lat_seq <- seq(lat_bound[1], lat_bound[2], length.out = n)

        bottom <- cbind(lon_seq, lat_bound[1])
        right <- cbind(rep(lon_bound[2], n - 2), lat_seq[2:(n - 1)])
        pole <- matrix(c(0, 90), ncol = 2)
        left <- cbind(rep(lon_bound[1], n - 2), rev(lat_seq[2:(n - 1)]))

        ring <- rbind(bottom, right, pole, left, bottom[1, ])
        st_sfc(st_polygon(list(ring)), crs = 4326)
    }

    stopifnot(length(vars) == length(row_labels))
    stopifnot(length(vars) == length(col_labels))

    if (is.null(row_order)) row_order <- unique(row_labels)
    if (is.null(col_order)) col_order <- unique(col_labels)

    crop_ll <- make_lonlat_crop(LON_BOUND, LAT_BOUND)

    coast <- ne_countries(scale = "medium", returnclass = "sf") |>
        st_union() |>
        st_intersection(crop_ll)

    graticule <- st_graticule(
        lat = GRAT_LAT,
        lon = GRAT_LON,
        xlim = c(-180, 180),
        ylim = c(-90, 90),
        crs = st_crs(4326)
    ) |>
        st_intersection(crop_ll)

    facet_key <- data.frame(
        variable = vars,
        row_lab = row_labels,
        col_lab = col_labels,
        stringsAsFactors = FALSE
    )

    plot_df <- df |>
        dplyr::select(lon, lat, dplyr::all_of(vars)) |>
        tidyr::pivot_longer(
            cols = dplyr::all_of(vars),
            names_to = "variable",
            values_to = value_name
        ) |>
        dplyr::left_join(facet_key, by = "variable") |>
        dplyr::filter(!is.na(.data[[value_name]])) |>
        dplyr::mutate(
            row_lab = factor(row_lab, levels = row_order),
            col_lab = factor(col_lab, levels = col_order)
        ) |>
        dplyr::filter(
            dplyr::between(lon, LON_BOUND[1], LON_BOUND[2]),
            dplyr::between(lat, LAT_BOUND[1], LAT_BOUND[2])
        )

    clims <- range(plot_df[[value_name]], na.rm = TRUE)
    crop_proj_bbox <- st_bbox(st_transform(crop_ll, CRS))

    if (is.null(tile_width)) {
        u_lon <- sort(unique(plot_df$lon))
        tile_width <- if (length(u_lon) > 1) min(diff(u_lon)) else 0.5
    }
    if (is.null(tile_height)) {
        u_lat <- sort(unique(plot_df$lat))
        tile_height <- if (length(u_lat) > 1) min(diff(u_lat)) else 0.5
    }

    p <- ggplot() +
        geom_tile(
            data = plot_df,
            aes(x = lon, y = lat, fill = .data[[value_name]]),
            width = tile_width,
            height = tile_height
        )

    if (!is.null(contour_var)) {
        contour_var <- rlang::ensym(contour_var)

        contour_df <- df |>
            dplyr::filter(
                dplyr::between(lon, LON_BOUND[1], LON_BOUND[2]),
                dplyr::between(lat, LAT_BOUND[1], LAT_BOUND[2])
            )

        p <- p +
            geom_contour(
                data = contour_df,
                aes(x = lon, y = lat, z = !!contour_var),
                binwidth = contour_binwidth,
                color = "darkgreen",
                linewidth = 0.4
            )
    }

    p +
        geom_sf(
            data = graticule,
            color = "grey60",
            linewidth = 0.25,
            linetype = "solid"
        ) +
        geom_sf(
            data = coast,
            fill = NA,
            color = "black",
            linewidth = 0.3
        ) +
        coord_sf(
            crs = CRS,
            xlim = unname(crop_proj_bbox[c("xmin", "xmax")]),
            ylim = unname(crop_proj_bbox[c("ymin", "ymax")]),
            expand = FALSE,
            default_crs = st_crs(4326),
            clip = "off"
        ) +
        facet_grid(row_lab ~ col_lab) +
        get_diverging_scale(clims) +
        labs(fill = TeX(colorbar_title)) +
        guides(
            fill = guide_colorbar(
                title.position = "left",
                barwidth = unit(15, "lines"),
                barheight = unit(0.5, "lines")
            )
        ) +
        THEME_PUB +
        theme(
            strip.placement = "outside",
            strip.text = element_text(face = "bold"),
            legend.position = "bottom",
            legend.direction = "horizontal",
            legend.box.margin = margin(0, 0, 0, 0),
            legend.margin = margin(0, 0, 0, 0),
            axis.text.x = element_blank(),
            axis.text.y = element_blank(),
            axis.title.x = element_blank(),
            axis.title.y = element_blank()
        )
}


In [84]:
df_reg$lag12_inflow <- df_reg$lag12_inflow - df_reg$lag0_inflow
df_reg$lag24_inflow <- df_reg$lag24_inflow - df_reg$lag0_inflow
df_reg$lag12_outflow <- df_reg$lag12_outflow - df_reg$lag0_outflow
df_reg$lag24_outflow <- df_reg$lag24_outflow - df_reg$lag0_outflow
df_reg$lag12_ascent <- df_reg$lag12_ascent - df_reg$lag0_ascent
df_reg$lag24_ascent <- df_reg$lag24_ascent - df_reg$lag0_ascent
df_reg$lag0_inflow <- 0
df_reg$lag0_outflow <- 0
df_reg$lag0_ascent <- 0


In [20]:
make_regime_df <- function(x, regime_name) {
    reg_idx <- match(regime_name, x$regimes)
    stopifnot(!is.na(reg_idx))

    # Important:
    # expand.grid(lon=..., lat=...) gives rows in the same order as as.vector(mat)
    # when mat has dimensions [lon, lat]
    df <- expand.grid(
        lon = x$lon,
        lat = x$lat
    )

    for (lag_name in x$lag_names) {
        lag_idx <- match(lag_name, x$lag_names)

        for (var_name in x$variables) {
            var_idx <- match(var_name, x$variables)

            col_name <- paste(lag_name, var_name, sep = "_")

            # slice is [lon, lat]
            mat <- x$climatology[, , lag_idx, reg_idx, var_idx]

            # correct ordering: lon varies fastest, then lat
            df[[col_name]] <- as.vector(mat)
        }
    }

    df
}

grid_key <- expand.grid(
    lag = wcb_clim$lag_names,
    variable = wcb_clim$variables,
    stringsAsFactors = FALSE
)

grid_key$var_name <- paste(grid_key$lag, grid_key$variable, sep = "_")

vars <- grid_key$var_name
row_labels <- grid_key$lag
col_labels <- grid_key$variable

col_order <- wcb_clim$variables

lag_label_map <- setNames(
    paste0(wcb_clim$lag_names, " (", wcb_clim$lag_hours, " h)"),
    wcb_clim$lag_names
)

row_labels_pretty <- unname(lag_label_map[grid_key$lag])
row_order_pretty <- unname(lag_label_map[wcb_clim$lag_names])


In [38]:
for (reg in wcb_clim$regimes) {
    cat("Regime:", reg, "\n")

    df_reg <- make_regime_df(wcb_clim, reg)

    # labels
    lag_label_map <- setNames(
        paste0(wcb_clim$lag_names, " (", wcb_clim$lag_hours, " h)"),
        wcb_clim$lag_names
    )

    row_order_pretty <- unname(lag_label_map[wcb_clim$lag_names])
    col_order <- wcb_clim$variables

    # plotting row -> contour source row
    contour_source_map <- c(
        lag0  = "lag0",
        lag12 = "lag12",
        lag24 = "lag24"
    )

    z_one <- z_comp |>
        dplyr::filter(as.character(wrname) == reg) |>
        dplyr::mutate(lag_name = as.character(lag_name))

    contour_long <- tidyr::expand_grid(
        lag_name = wcb_clim$lag_names,
        col_lab = wcb_clim$variables
    ) |>
        dplyr::mutate(
            contour_lag = unname(contour_source_map[lag_name])
        ) |>
        dplyr::left_join(
            z_one |>
                dplyr::select(lon, lat, lag_name, z) |>
                dplyr::rename(contour_lag = lag_name),
            by = "contour_lag"
        ) |>
        dplyr::mutate(
            row_lab = lag_label_map[lag_name],
            row_lab = factor(row_lab, levels = row_order_pretty),
            col_lab = factor(col_lab, levels = col_order)
        ) |>
        dplyr::rename(contour_value = z)

    # mask contours: use the long df returned by generate_wcb_masks_df()
    mask_long <- masks |>
        dplyr::filter(regime == reg) |>
        dplyr::mutate(
            row_lab = lag_label_map[lag_name],
            row_lab = factor(row_lab, levels = row_order_pretty),
            col_lab = factor(variable, levels = col_order),
            mask_bin = as.integer(mask_value > 0)
        )

    p <- plot_composite_map_grid(
        df = df_reg,
        vars = vars,
        row_labels = row_labels_pretty,
        col_labels = col_labels,
        row_order = row_order_pretty,
        col_order = col_order,
        contour_var = NULL,
        colorbar_title = "Climatology"
    ) +
        geom_contour(
            data = contour_long,
            aes(x = lon, y = lat, z = contour_value),
            color = "darkgreen",
            linewidth = 0.4,
            binwidth = 80 * 9.80665,
            inherit.aes = FALSE
        ) +
        geom_contour(
            data = mask_long,
            aes(x = lon, y = lat, z = mask_bin),
            color = "darkblue",
            linewidth = 0.5,
            breaks = 0.5,
            inherit.aes = FALSE
        ) +
        scale_fill_gradient(
            low = "white",
            high = "darkred",
            limits = c(0, max(unlist(df_reg[vars]), na.rm = TRUE)),
            oob = scales::squish
        ) +
        labs(fill = "Climatology")

    save_plot(p, paste0("WCB_clim_", reg, ".pdf"), width = 7, height = 7)
}


Regime: AT 


Warning message in dplyr::left_join(dplyr::mutate(tidyr::expand_grid(lag_name = wcb_clim$lag_names, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `d

Regime: ZO 


Warning message in dplyr::left_join(dplyr::mutate(tidyr::expand_grid(lag_name = wcb_clim$lag_names, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `d

Regime: ScTr 


...

Warning message in dplyr::left_join(dplyr::mutate(tidyr::expand_grid(lag_name = wcb_clim$lag_names, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `d

Regime: GL 


Warning message in dplyr::left_join(dplyr::mutate(tidyr::expand_grid(lag_name = wcb_clim$lag_names, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `d

Regime: no 


Warning message in dplyr::left_join(dplyr::mutate(tidyr::expand_grid(lag_name = wcb_clim$lag_names, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `default_crs = NULL`.”
Warning message:
“Projection of x or y limits failed in `coord_sf()`.
ℹ Consider setting `lims_method = "geometry_bbox"` or `d

# Find masks for each regime

In [36]:
generate_wcb_masks_df <- function(data, percent = 0.8, return_weighted = TRUE) {
    lon <- data$lon
    lat <- data$lat
    lats_rad <- lat * pi / 180
    regimes <- data$regimes
    variables <- data$variables
    lag_names <- data$lag_names
    climatology <- data$climatology

    # precompute cosine weights for latitude (for coverage calculation only)
    lat_weights <- cos(lats_rad)
    lat_mat <- matrix(lat_weights, nrow = length(lon), ncol = length(lat), byrow = TRUE)

    out_list <- list()
    k <- 1

    for (var_idx in seq_along(variables)) {
        var_name <- variables[var_idx]

        for (reg_idx in seq_along(regimes)) {
            reg_name <- regimes[reg_idx]

            for (lag_idx in seq_along(lag_names)) {
                lag_name <- lag_names[lag_idx]

                # extract 2D field for this variable, regime, lag
                freq_field <- climatology[, , lag_idx, reg_idx, var_idx]

                # flatten raw values (for ranking)
                freq_vec <- as.vector(freq_field)
                idx_flat <- order(freq_vec, decreasing = TRUE)

                # compute cumulative weighted sum (lat weighting applied here)
                weighted_vec <- as.vector(freq_field * lat_mat)
                cum_weight <- cumsum(weighted_vec[idx_flat])
                total_weight <- sum(weighted_vec, na.rm = TRUE)

                # number of points to cover `percent` of total weighted sum
                n_sel <- which(cum_weight / total_weight >= percent)[1]
                selected_idx <- idx_flat[1:n_sel]

                # create mask matrix: put original freq values at selected indices
                mask_mat <- matrix(0, nrow = length(lon), ncol = length(lat))
                mask_mat[selected_idx] <- freq_vec[selected_idx]

                # normalize if requested
                if (return_weighted) {
                    mask_mat <- mask_mat / sum(mask_mat)
                } else {
                    mask_mat[mask_mat > 0] <- 1
                }

                # convert to long data.frame
                grid <- expand.grid(lon = lon, lat = lat)
                out_list[[k]] <- data.frame(
                    lon = grid$lon,
                    lat = grid$lat,
                    variable = var_name,
                    regime = reg_name,
                    lag_name = lag_name,
                    mask_value = as.vector(mask_mat)
                )
                k <- k + 1
            }
        }
    }

    dplyr::bind_rows(out_list)
}


In [37]:
masks <- generate_wcb_masks_df(wcb_clim, percent = 0.33, return_weighted = TRUE)


In [27]:
str(masks)


'data.frame':	522720 obs. of  6 variables:
 $ lon       : num  -80 -79 -78 -77 -76 -75 -74 -73 -72 -71 ...
 $ lat       : num  30 30 30 30 30 30 30 30 30 30 ...
 $ variable  : chr  "inflow" "inflow" "inflow" "inflow" ...
 $ regime    : chr  "AT" "AT" "AT" "AT" ...
 $ lag_name  : chr  "lag0" "lag0" "lag0" "lag0" ...
 $ mask_value: num  0.00256 0.00291 0.00326 0.00359 0.00359 ...


In [39]:
saveRDS(masks, "../data/wcb_masks_df.rds")
